# RSA threat demo — factor 15 via order-finding (PennyLane)

Shor reduces factoring to period-finding. For $N=15$, $a=7$ the period is 4,
giving factors $\gcd(7^2 \pm 1, 15) = (3,5)$.

In [ ]:
import pennylane as qml
import numpy as np
from math import gcd

N_CTRL = 4
N_TARGET = 4
N_TOTAL = N_CTRL + N_TARGET
dev = qml.device("default.qubit", wires=N_TOTAL)

## Period-finding circuit

In [ ]:
def controlled_mult_by_a(a, power, ctrl_wires, target_wires):
    for _ in range(power):
        for i, tw in enumerate(target_wires):
            qml.ctrl(qml.Pow(qml.PauliX(wires=tw), a), control=ctrl_wires[i])


@qml.qnode(dev)
def period_finding_circuit(a):
    ctrl = list(range(N_CTRL))
    tgt = list(range(N_CTRL, N_TOTAL))
    qml.Hadamard(wires=ctrl)
    qml.PauliX(wires=tgt[0])
    qml.ctrl(controlled_mult_by_a, control=ctrl)(a, 1, ctrl, tgt)
    qml.adjoint(qml.QFT(wires=ctrl))
    return qml.probs(wires=ctrl)

a = 7
print(qml.draw(period_finding_circuit)(a))

In [ ]:
probs = period_finding_circuit(a)
measured = np.argmax(probs)
phase = measured / (2**N_CTRL)
print(f"measured: {measured}  phase ~ {phase:.6f}")
r = int(round(2**N_CTRL * phase))
if r > 0 and a ** r % 15 == 1:
    print(f"period r = {r}")
    print(f"{a}^{r} mod 15 = {pow(a, r, 15)}")

In [ ]:
r = 4
x = pow(a, r // 2, 15)
f1 = gcd(x - 1, 15)
f2 = gcd(x + 1, 15)
print(f"{15} = {f1} x {f2}")